In [1]:
import pandas as pd

# LOADING THE DATASETS
print("Loading datasets...")
df_2nd = pd.read_excel('2nd_families_evaluated.xlsx')
df_3rd = pd.read_excel('3rd_families_evaluated.xlsx')
df_master = pd.read_excel('master_dataset_with_ai_features.xlsx')

# MERGING EVALUATIONS TO FIND CONFLICTS
# We only need the ID and the status from both evaluated datasets to find the conflicts
merged_eval = pd.merge(
    df_2nd[['hasfamilyid', 'Predicted_Status']], 
    df_3rd[['hasfamilyid', 'Predicted_Status']], 
    on='hasfamilyid', 
    suffixes=('_2nd', '_3rd')
)

# DEFINING CONDITIONS & CALCULATE COUNTS
cond_0_0 = (merged_eval['Predicted_Status_2nd'] == 0) & (merged_eval['Predicted_Status_3rd'] == 0)
cond_0_1 = (merged_eval['Predicted_Status_2nd'] == 0) & (merged_eval['Predicted_Status_3rd'] == 1)
cond_1_0 = (merged_eval['Predicted_Status_2nd'] == 1) & (merged_eval['Predicted_Status_3rd'] == 0)
cond_1_1 = (merged_eval['Predicted_Status_2nd'] == 1) & (merged_eval['Predicted_Status_3rd'] == 1)

print("\n--- Model Prediction Comparison Results ---")
print(f"Total matching families found: {len(merged_eval)}\n")
print(f"1. 2nd Eval (0) and 3rd Eval (0): {len(merged_eval[cond_0_0])} families")
print(f"2. 2nd Eval (0) and 3rd Eval (1): {len(merged_eval[cond_0_1])} families")
print(f"3. 2nd Eval (1) and 3rd Eval (0): {len(merged_eval[cond_1_0])} families  <-- (Flagged for Verification)")
print(f"4. 2nd Eval (1) and 3rd Eval (1): {len(merged_eval[cond_1_1])} families")

print("\n--- Summary Matrix (Cross-Tabulation) ---")
print(pd.crosstab(merged_eval['Predicted_Status_2nd'], merged_eval['Predicted_Status_3rd'], 
                  rownames=['2nd_Evaluated'], colnames=['3rd_Evaluated']))

# EXTRACTING MASTER FEATURES FOR FLAGGED IDs
# Getting the exact list of Family IDs that fall under Condition 3
flagged_ids = merged_eval.loc[cond_1_0, 'hasfamilyid']

# Filtering the MASTER dataset to only include these flagged families
verification_df = df_master[df_master['hasfamilyid'].isin(flagged_ids)].copy()

# ATTACHING 3RD MODEL SCORES & EXPORTING
# We isolate the specific columns we want to bring over from the 3rd evaluation.
# We rename them to be explicit in the final output file.
scores_to_attach = df_3rd[['hasfamilyid', 'Predicted_Status', 'BPL_Probability_Score']].rename(
    columns={
        'Predicted_Status': 'Predicted_Status_3rd',
        'BPL_Probability_Score': 'BPL_Probability_Score_3rd'
    }
)

# Merging the scores onto our filtered master features using a Left Join
final_verification_df = pd.merge(
    verification_df,
    scores_to_attach,
    on='hasfamilyid',
    how='left'
)

# Saving to Excel
output_filename = 'Need_for_verification.xlsx'
final_verification_df.to_excel(output_filename, index=False)

print(f"\n✅ Success! Extracted master features and scores for {len(final_verification_df)} conflicting families.")
print(f"Exported to '{output_filename}' for manual review.")

Loading datasets...

--- Model Prediction Comparison Results ---
Total matching families found: 2301

1. 2nd Eval (0) and 3rd Eval (0): 976 families
2. 2nd Eval (0) and 3rd Eval (1): 7 families
3. 2nd Eval (1) and 3rd Eval (0): 1104 families  <-- (Flagged for Verification)
4. 2nd Eval (1) and 3rd Eval (1): 214 families

--- Summary Matrix (Cross-Tabulation) ---
3rd_Evaluated     0    1
2nd_Evaluated           
0               976    7
1              1104  214

✅ Success! Extracted master features and scores for 1104 conflicting families.
Exported to 'Need_for_verification.xlsx' for manual review.


In [2]:
import pandas as pd

# Loading the file with the 1104 flagged families
print("Loading verification dataset...")
df = pd.read_excel('Need_for_verification.xlsx')

# CALCULATING NON-BPL PROBABILITY
# Since your BPL score is on a 0-100 scale, we subtract from 100
df['Non_BPL_Probability'] = 100 - df['BPL_Probability_Score_3rd']

# SEPARATE INTO THE TWO GROUPS
# Group 1: Greater than 80% confident NON-BPL
high_confidence_df = df[df['Non_BPL_Probability'] > 80].copy()
high_confidence_df = high_confidence_df.sort_values(by='Non_BPL_Probability', ascending=False)

# Group 2: Between 50% and 80% confident NON-BPL
# (Using <= 80 to catch exactly 80.0%, and > 50 because they are all inherently > 50)
borderline_df = df[(df['Non_BPL_Probability'] > 50) & (df['Non_BPL_Probability'] <= 80)].copy()
borderline_df = borderline_df.sort_values(by='Non_BPL_Probability', ascending=True)

# PRINTING THE RESULTS & EXPORTING
print("\n--- AI Confidence Breakdown ---")
print(f"Total families analyzed: {len(df)}")
print(f"1. High Confidence Non-BPL (>80%): {len(high_confidence_df)} families")
print(f"2. Borderline Non-BPL (50%-80%): {len(borderline_df)} families")

# Saving them to separate sheets or files
high_confidence_df.to_excel('High_Confidence_Non_BPL.xlsx', index=False)
borderline_df.to_excel('Borderline_Non_BPL.xlsx', index=False)

print("\n✅ Success! Exported both groups into separate Excel files.")

Loading verification dataset...

--- AI Confidence Breakdown ---
Total families analyzed: 1104
1. High Confidence Non-BPL (>80%): 1007 families
2. Borderline Non-BPL (50%-80%): 97 families

✅ Success! Exported both groups into separate Excel files.
